# Explore ADS `cf_request_log` with DuckDB Python

Mirrors **Overview → Connection Details → DuckDB Python** (`quack:` ATTACH).

**Setup**
1. Python 3.11+ venv: `python3.11 -m venv .venv-explore && .venv-explore/bin/pip install -r requirements-explore.txt`
2. In `dashboard/.env` set `ADS_QUERY_HOST` and `ADS_QUACK_TOKEN` (Overview → **Show token**).
3. Select the `.venv-explore` kernel for this notebook.

The FastAPI dashboard keeps using HTTP `POST /query` + `DIGITALOCEAN_TOKEN`. This notebook is for interactive exploration (pandas, local joins).

In [ ]:
import os
from pathlib import Path

import duckdb
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

HOST = os.environ["ADS_QUERY_HOST"].removeprefix("https://").rstrip("/")
if ":" not in HOST:
    HOST = f"{HOST}:443"
TOKEN = os.environ["ADS_QUACK_TOKEN"]
TABLE = os.environ.get("ADS_TABLE", "tabular.ads_cloudflare_poc.cf_request_log")

print("duckdb", duckdb.__version__, "host", HOST)

In [ ]:
con = duckdb.connect()
con.execute("INSTALL quack; LOAD quack;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE quack, TOKEN '{TOKEN}');")
con.execute(f"ATTACH 'quack:{HOST}' AS srv;")
con.execute("SHOW DATABASES").df()

In [ ]:
# One-minute window — same idea as the dashboard "1 minute" preset
df = con.execute(f"""
SELECT
  count(*) AS rows,
  min(edge_start_timestamp) AS t_min,
  max(edge_start_timestamp) AS t_max
FROM srv.{TABLE}
WHERE edge_start_timestamp >= TIMESTAMP '2026-11-10 14:30:00 UTC'
  AND edge_start_timestamp <  TIMESTAMP '2026-11-10 14:31:00 UTC'
""").df()
df

In [ ]:
hosts = con.execute(f"""
SELECT split_part(client_request_host, ':', 1) AS host, count(*) AS n
FROM srv.{TABLE}
WHERE edge_start_timestamp >= TIMESTAMP '2026-11-10 14:30:00 UTC'
  AND edge_start_timestamp <  TIMESTAMP '2026-11-10 14:31:00 UTC'
GROUP BY 1
ORDER BY n DESC
LIMIT 15
""").df()
hosts

## Optional: join remote ADS rows with a local DataFrame

This is the main benefit of DuckDB Python vs the HTTP dashboard path.

In [ ]:
import pandas as pd

local = pd.DataFrame({"host": ["api.digitalocean.com", "mcp.digitalocean.com"], "team": ["API", "MCP"]})
con.register("local_hosts", local)
con.execute(f"""
SELECT l.team, r.host, r.n
FROM local_hosts l
JOIN (
  SELECT split_part(client_request_host, ':', 1) AS host, count(*) AS n
  FROM srv.{TABLE}
  WHERE edge_start_timestamp >= TIMESTAMP '2026-11-10 14:30:00 UTC'
    AND edge_start_timestamp <  TIMESTAMP '2026-11-10 14:31:00 UTC'
  GROUP BY 1
) r USING (host)
ORDER BY r.n DESC
""").df()